# 沪深300历史样本质量复核（CR-013）
## tl;dr
6只腾讯真实样本均返回；5只完整242日，688981为241日，缺2025-09-08。上交所2025-024公告证实该日停牌，9月9日复牌。自动覆盖器尚不消费停牌证据，故候选仍gaps且不入可用库；不能称全300通过。

## Context & Methods
颗粒度：固定名单版本+股票身份+qfq+区间。主键：symbol/adjust/trade_date。检查来源哈希、完整Schema、交易日集合、重复与非法OHLCV、金额空值、隔离库重读及零网络重放一致性。
### Key Assumptions
当前名单用于历史研究有幸存者偏差；242日是市场交易日，不等于每只股票可交易日。观察文件是SDK与项目适配后的数据，不是上游原始HTTP字节；基本质量检查不证明除权复权经济正确、量额单位跨源一致或许可。

## Data
只读artifacts/cr013-history-20260908与其离线重放。合成测试不作为真实覆盖证据；不触及生产数据库。

In [ ]:
import json, sqlite3, sys
from pathlib import Path
from datetime import date
from collections import Counter
import pandas as pd, yaml
from jsonschema import Draft202012Validator, FormatChecker
root = Path.cwd()
sys.path.insert(0, str(root / 'services/algorithms/src'))
from quant_platform.data.coverage import digest, assess_history
candidate = root / 'artifacts/cr013-history-20260908'
manifest = json.loads((candidate / 'manifest.json').read_text())
replay = json.loads((root / 'artifacts/cr013-history-replay-20260908/manifest.json').read_text())
schema = yaml.safe_load((root / 'packages/contracts/schemas/coverage.yaml').read_text())
validator = Draft202012Validator({'$ref': '#/HistoryCandidate', **schema}, format_checker=FormatChecker())
for item in (manifest, replay):
    validator.validate(item)
    assert item['candidateId'] == digest({k: v for k, v in item.items() if k != 'candidateId'})
assert manifest['entries'] == replay['entries']
assert replay['networkRequestsThisRun'] == 0
assert len({e['assetId'] for e in manifest['entries']}) == 300
assert dict(Counter(e['status'] for e in manifest['entries'])) == manifest['statusCounts']
print({k: manifest[k] for k in ['universeVersion', 'startDate', 'endDate', 'createdAt', 'statusCounts', 'networkRequestsThisRun']})

## Results
逐只重算质量；通过样本逐值核对隔离SQLite，缺口样本确认未入库。

In [ ]:
start, end = date.fromisoformat(manifest['startDate']), date.fromisoformat(manifest['endDate'])
rows, requests = [], 0
conn = sqlite3.connect((candidate / 'candidate-cache/market_data.db').as_uri() + '?mode=ro', uri=True)
try:
    for entry in manifest['entries']:
        if entry['status'] == 'not_attempted': continue
        wrapper = json.loads((candidate / 'observations' / (entry['symbol'] + '.json')).read_text())
        value = wrapper['observation']
        assert digest(value) == wrapper['sha256'] == entry['observationSha256']
        assert wrapper['universeVersion'] == manifest['universeVersion']
        assert value['error'] is None
        requests += len(value['httpTrace'])
        assert all(e['status'] == 200 and e['error'] is None for e in value['httpTrace'])
        frame = pd.DataFrame(value['records'])
        quality = assess_history(frame, start, end)
        assert quality == entry['quality']
        stored = pd.read_sql_query('SELECT trade_date AS date, open, high, low, close, volume, amount FROM ohlcv WHERE symbol=? AND adjust=? ORDER BY trade_date', conn, params=(entry['symbol'], 'qfq'))
        if quality['status'] == 'complete':
            frame['date'] = pd.to_datetime(frame['date'])
            stored['date'] = pd.to_datetime(stored['date'])
            pd.testing.assert_frame_equal(frame[stored.columns], stored, check_dtype=False)
        else:
            assert stored.empty
        rows.append({'symbol': entry['symbol'], 'status': quality['status'], 'rows': quality['rowCount'], 'expected': quality['expectedSessions'], 'missing': quality['missingSessions'], 'invalid': quality['invalidRows'], 'duplicates': quality['duplicateRows'], 'amountMissing': quality['amountMissingRows']})
finally:
    conn.close()
assert requests == manifest['networkRequestsThisRun'] == 18
pd.DataFrame(rows)

## Takeaways
当前已验证5/300完整市场交易日覆盖，1只缺口已获得停牌解释，294只未采集。下一步实现证据化停牌/上市/代码变更分类与研究可交易日约束，再分批扩大；不能填价、缩小分母或把停牌当成服务器故障。本批只采集一次、一个环境，不能判断跨日稳定性或历史版本漂移。
官方停复牌证据（人工核查，未被上述代码自动导入）：[中芯国际2025-024公告，2025-09-09](https://star.sse.com.cn/disclosure/listedinfo/announcement/c/new/2025-09-09/688981_20250909_2BQ9.pdf)。